# Phase 5 — Junior Data Scientist Skill Modeling & Interpretability
### SAS Hackathon Round 2: InsightPath / RUSTY WOLVES

This notebook executes the supervised predictive modeling layer specifically for the Junior Data Scientist (JDS) cohort ($N=139$).
It addresses **Research Question RQ8**:
> *"Can interpretable classification models predict observed junior salary-hike classifications from technical skill variables?"*

### Core Methodological Principles:
1. **Leakage-Free Cross-Validation**: 5-Fold x 5-Repeat Stratified Cross-Validation (25 validation splits, `random_state=42`). Preprocessing (`StandardScaler`) is encapsulated inside pipelines.
2. **Parsimony Over Complexity**: Small-sample discipline ($N=139$). We evaluate transparent, regularized models against constrained tree ensembles.
3. **Dual Metric Evaluation**: Primary metrics are **ROC-AUC** and **Macro F1** evaluated on held-out validation folds.
4. **Out-of-Sample Interpretability**: Standardized odds ratios, visual decision rules, and validation permutation feature importance.
5. **Sensitivity Robustness**: Out-of-sample comparison against the anomaly-free sensitivity cohort ($N=137$, ID 3291 excluded).


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path
sys.path.append(str(Path(".").resolve().parent.parent))

from src.modeling.jds.data import load_jds_primary, load_jds_sensitivity, audit_jds_anomalies, JDS_FEATURES, JDS_REDUCED_FEATURES
from src.modeling.jds.pipelines import build_candidate_pipelines, build_reduced_pipelines
from src.modeling.jds.cross_validation import evaluate_pipelines_cv
from src.modeling.jds.evaluation import build_model_comparison_table, compute_oof_confusion_matrix
from src.modeling.jds.interpretability import extract_logistic_interpretability, extract_tree_rules, compute_validation_permutation_importance

print("Libraries and Phase 5 modules loaded successfully.")


## 1. Data Ingestion & Dataset Validation
Load primary ($N=139$) and sensitivity ($N=137$) analytical datasets. Verify target and feature matrix isolation.

In [ ]:
base_dir = Path("../..")
X_prim, y_prim, df_prim = load_jds_primary(base_dir)
X_sens, y_sens, df_sens = load_jds_sensitivity(base_dir)

print(f"Primary Dataset: N={len(X_prim)}, Features={list(X_prim.columns)}")
print(f"Target Balance: Class 1 (High Hike) = {(y_prim == 1).sum()} ({(y_prim == 1).mean()*100:.2f}%), Class 0 = {(y_prim == 0).sum()} ({(y_prim == 0).mean()*100:.2f}%)")
print(f"Sensitivity Dataset: N={len(X_sens)} (ID 3291 excluded)")


## 2. Duplicate & Anomaly Audit (ID 3291)
Inspect the contradictory duplicate record identified in Phase 1 and evaluated in Phase 4.

In [ ]:
audit_res = audit_jds_anomalies(df_prim, df_sens)
for k, v in audit_res.items():
    print(f"{k}: {v}")


## 3. Naive Majority Baseline Benchmark
Zero-rule classifier predicting majority class (High Hike) yields base accuracy of 52.52% and ROC-AUC of 0.500.

In [ ]:
df_base = pd.read_csv("../../outputs/tables/phase5/phase5_baseline_metrics.csv")
display(df_base)


## 4. Repeated Stratified Cross-Validation (25 Splits)
Evaluate candidate classification pipelines across 5 folds x 5 repeats.

In [ ]:
df_perf = pd.read_csv("../../outputs/tables/phase5/phase5_model_performance.csv")
display(df_perf[["model_name", "roc_auc_mean", "roc_auc_std", "macro_f1_mean", "accuracy_mean", "balanced_accuracy_mean"]])


## 5. Candidate Model Comparison & Selection Scorecard
Rank candidate models by ROC-AUC, Macro F1, stability, and interpretability.

In [ ]:
df_comp = pd.read_csv("../../outputs/tables/phase5/phase5_model_comparison.csv")
display(df_comp[["rank", "model_name", "roc_auc_mean", "macro_f1_mean", "accuracy_mean", "accuracy_lift_pct_points", "interpretability", "model_complexity"]])


## 6. Champion Model Interpretability (Logistic L2 Standardized Odds Ratios)
Extract standardized odds ratios and assess coefficient stability across all 25 splits.

In [ ]:
df_or = pd.read_csv("../../outputs/tables/phase5/phase5_logistic_odds_ratios.csv")
df_stab = pd.read_csv("../../outputs/tables/phase5/phase5_logistic_stability.csv")
display(df_or)
display(df_stab[["feature_label", "mean_coef_beta", "std_coef_beta", "mean_odds_ratio", "sign_consistency_pct", "stability_status"]])


## 7. Decision Tree Rule Extraction (CART Depth <= 3)
Inspect human-readable transparent decision pathways extracted from the tree.

In [ ]:
df_rules = pd.read_csv("../../outputs/tables/phase5/phase5_tree_rules.csv")
display(df_rules[["rule_id", "decision_path", "outcome_label", "leaf_samples_n", "prob_high_hike"]])


## 8. Random Forest Validation Permutation Importance
Permutation importance evaluated strictly on held-out validation folds across 25 splits.

In [ ]:
df_rf = pd.read_csv("../../outputs/tables/phase5/phase5_rf_permutation_importance.csv")
display(df_rf[["importance_rank", "feature_label", "mean_permutation_importance", "std_permutation_importance", "ci_95_lower", "ci_95_upper"]])


## 9. Out-of-Fold Error Analysis
Examine false positives, false negatives, and boundary uncertainty ($0.35 \le p \le 0.65$).

In [ ]:
df_err = pd.read_csv("../../outputs/tables/phase5/phase5_error_analysis.csv")
display(df_err)


## 10. Parsimony Analysis: Full 5-Feature vs Reduced 2-Feature Models
Evaluate whether Maths/Stats + Storytelling alone preserve predictive performance.

In [ ]:
df_pars = pd.read_csv("../../outputs/tables/phase5/phase5_full_vs_reduced_features.csv")
display(df_pars[["algorithm_family", "full_roc_auc", "reduced_roc_auc", "delta_roc_auc", "pct_auc_retained", "parsimony_assessment"]])


## 11. Sensitivity Modeling on N=137 (Robustness Audit)
Confirm that performance and rankings are invariant to excluding contradictory ID 3291.

In [ ]:
df_sens_p = pd.read_csv("../../outputs/tables/phase5/phase5_sensitivity_model_performance.csv")
display(df_sens_p[["model_name", "primary_roc_auc", "sensitivity_roc_auc", "delta_roc_auc", "robustness_verdict"]])


## 12. Phase 4 Inferential vs Phase 5 Predictive Traceability
Synthesize Phase 4 statistical decisions with Phase 5 machine-learning evidence.

In [ ]:
df_trace = pd.read_csv("../../outputs/tables/phase5/phase5_phase4_traceability.csv")
display(df_trace[["feature", "phase4_inferential_finding", "phase5_predictive_validation", "concordance_verdict", "substantive_interpretation"]])
